In [23]:
import pandas as pd
from scipy.stats import wilcoxon
from pathlib import Path

# === 1. Load your CSVs ===
# Update the filenames to match yours
path_cnn = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/MTB-CNN/training_output/results_ccp_filter12_epoch250_sbatch_seed_1_cv0/cv_split_0_auc.csv")
path_dnabert = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/zero_shot/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")
# path_dnabert = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/finetune/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

df_cnn = pd.read_csv(path_cnn)
df_dnabert = pd.read_csv(path_dnabert)

# === 2. Keep only relevant columns ===
# Adjust names to match your file (case-sensitive)
df_cnn = df_cnn[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD-CNN"})
df_dnabert = df_dnabert[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD-DNABERT"})

# === 3. Merge on Drug and average across folds ===
# If there are multiple folds per drug, this will average them before comparison
per_drug = (
    pd.merge(df_cnn, df_dnabert, on="Drug", how="inner")
    .groupby("Drug")[["AUC_MD-CNN", "AUC_MD-DNABERT"]]
    .mean()
    .dropna()
)

# === 4. Wilcoxon signed-rank test ===
# stat, p = wilcoxon(
#     per_drug["AUC_MD-DNABERT"],
#     per_drug["AUC_MD-CNN"],
#     alternative="less"  # MD-DNABERTCNN > MD-CNN
# )

stat, p = wilcoxon(
    per_drug["AUC_MD-DNABERT"],
    per_drug["AUC_MD-CNN"],
    alternative="two-sided", mode="approx"  # MD-DNABERTCNN > MD-CNN
)

print(f"Wilcoxon signed-rank test result:")
print(f"Statistic = {stat:.3f}, p-value = {p:.4f}")

# === 5. Effect size summary ===
per_drug["ΔAUC"] = per_drug["AUC_MD-DNABERT"] - per_drug["AUC_MD-CNN"]
median_diff = per_drug["ΔAUC"].median()
mean_diff = per_drug["ΔAUC"].mean()

print(f"\nMedian ΔAUC = {median_diff:.3f}")
print(f"Mean   ΔAUC = {mean_diff:.3f}")
print(f"Number of drugs compared = {len(per_drug)}")


Wilcoxon signed-rank test result:
Statistic = 1.000, p-value = 0.0044

Median ΔAUC = -0.090
Mean   ΔAUC = -0.122
Number of drugs compared = 11


In [9]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/zero_shot/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

# Compute overall mean AUC
overall_mean_auc = df["AUC"].mean()

print(f"Overall mean AUC = {overall_mean_auc:.4f}")


Overall mean AUC = 0.8244


### Per drug level paired signifiance test

In [18]:
import pandas as pd
from scipy.stats import wilcoxon
from pathlib import Path
import numpy as np

# === 1. Paths to both model CSVs ===
path_modelA = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/MTB-CNN/training_output/results_ccp_filter12_epoch250_sbatch_seed_1_cv0/auc.csv")
path_modelB = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/finetune/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

# === 2. Read both ===
df_A = pd.read_csv(path_modelA)
df_B = pd.read_csv(path_modelB)

# === 3. Keep only relevant columns ===
df_A = df_A[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_CNN"})
df_B = df_B[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_DNABERT"})

modelA_name = "MD-CNN"
modelB_name = "MD-DNABERT"

# === 4. Get intersection of drugs ===
drugs = sorted(set(df_A["Drug"].unique()) & set(df_B["Drug"].unique()))
print(f"✅ Found {len(drugs)} drugs in both CSVs: {drugs}\n")

results = []

# === 5. Compute per-drug Wilcoxon test ===
for drug in drugs:
    auc_a = df_A.loc[df_A["Drug"] == drug, "AUC_MD_CNN"].values
    auc_b = df_B.loc[df_B["Drug"] == drug, "AUC_MD_DNABERT"].values

    # Broadcast single-fold MD-CNN values to match DNABERT folds
    # if len(auc_a) == 1 and len(auc_b) > 1:
    #     auc_a = np.repeat(auc_a, len(auc_b))
    #     # print(f"🔁 Broadcasted {drug}: 1→{len(auc_b)} folds for MD-CNN")

    if len(auc_a) == len(auc_b) and len(auc_a) > 1:
        try:
            stat, p = wilcoxon(auc_a, auc_b, alternative="greater")  # test if MD-CNN > DNABERT
        except ValueError:
            stat, p = (None, None)

        results.append({
            "Drug": drug,
            f"Mean_{modelA_name}": auc_a.mean(),
            f"Mean_{modelB_name}": auc_b.mean(),
            "ΔAUC": auc_a.mean() - auc_b.mean(),
            "Wilcoxon_stat": stat,
            "p_value": p
        })
    else:
        print(f"⚠️ Skipping {drug}: unmatched folds even after broadcast.")

# === 6. Convert to DataFrame and save ===
if len(results) == 0:
    print("\n❌ No valid entries — check Drug names or CSV structure.")
else:
    res_df = pd.DataFrame(results).sort_values("p_value", na_position="last")
    res_df["Significant (p<0.05)"] = res_df["p_value"] < 0.05

    print("\n✅ Per-drug Wilcoxon significance test results:")
    print(res_df.to_string(index=False, float_format="%.4f"))

    # out_path = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/stat_tests/wilcoxon_MD-CNN_vs_MD-DNABERT.csv")
    # res_df.to_csv(out_path, index=False)
    # print(f"\n💾 Saved results to: {out_path}")


✅ Found 11 drugs in both CSVs: ['AMIKACIN', 'CAPREOMYCIN', 'ETHAMBUTOL', 'ETHIONAMIDE', 'ISONIAZID', 'KANAMYCIN', 'LEVOFLOXACIN', 'MOXIFLOXACIN', 'PYRAZINAMIDE', 'RIFAMPICIN', 'STREPTOMYCIN']


✅ Per-drug Wilcoxon significance test results:
        Drug  Mean_MD-CNN  Mean_MD-DNABERT    ΔAUC  Wilcoxon_stat  p_value  Significant (p<0.05)
    AMIKACIN       0.9039           0.4121  0.4918        15.0000   0.0312                  True
 CAPREOMYCIN       0.8847           0.8061  0.0786        15.0000   0.0312                  True
   KANAMYCIN       0.9346           0.8659  0.0688        15.0000   0.0312                  True
   ISONIAZID       0.9685           0.8094  0.1591        15.0000   0.0312                  True
LEVOFLOXACIN       0.9370           0.8625  0.0746        15.0000   0.0312                  True
PYRAZINAMIDE       0.9205           0.6592  0.2613        15.0000   0.0312                  True
  RIFAMPICIN       0.9736           0.7519  0.2217        15.0000   0.0312      

In [33]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from pathlib import Path
from statsmodels.stats.multitest import multipletests

# === 1. Paths to both model CSVs ===
path_modelA = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/MTB-CNN/training_output/results_ccp_filter12_epoch250_sbatch_seed_1_cv0/auc.csv")
path_modelB = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/finetune/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

# === 2. Read both ===
df_A = pd.read_csv(path_modelA)
df_B = pd.read_csv(path_modelB)

# === 3. Keep only relevant columns ===
df_A = df_A[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_CNN"})
df_B = df_B[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_DNABERT"})

modelA_name = "MD-CNN"
modelB_name = "MD-DNABERT"

# === 4. Get intersection of drugs ===
drugs = sorted(set(df_A["Drug"].unique()) & set(df_B["Drug"].unique()))
print(f"✅ Found {len(drugs)} drugs in both CSVs: {drugs}\n")

results = []

# === 5. Compute per-drug Mann–Whitney U test ===
# for drug in drugs:
#     auc_a = df_A.loc[df_A["Drug"] == drug, "AUC_MD_CNN"].values
#     auc_b = df_B.loc[df_B["Drug"] == drug, "AUC_MD_DNABERT"].values

#     # Skip if either group has <2 samples
#     if len(auc_a) < 2 or len(auc_b) < 2:
#         print(f"⚠️ Skipping {drug}: too few samples ({len(auc_a)} vs {len(auc_b)})")
#         continue

#     try:
#         # alternative="greater" tests if AUC_MD_CNN > AUC_MD_DNABERT
#         stat, p = mannwhitneyu(auc_a, auc_b, alternative="greater", method="auto")
#     except ValueError:
#         stat, p = (np.nan, np.nan)

#     results.append({
#         "Drug": drug,
#         f"Mean_{modelA_name}": np.mean(auc_a),
#         f"Mean_{modelB_name}": np.mean(auc_b),
#         "ΔAUC": np.mean(auc_a) - np.mean(auc_b),
#         "MannWhitneyU_stat": stat,
#         "p_value": p
#     })

from scipy.stats import mannwhitneyu

for drug in drugs:
    auc_a = df_A.loc[df_A["Drug"] == drug, "AUC_MD_CNN"].values
    auc_b = df_B.loc[df_B["Drug"] == drug, "AUC_MD_DNABERT"].values

    if len(auc_a) < 2 or len(auc_b) < 2:
        print(f"⚠️ Skipping {drug}: too few samples")
        continue

    try:
        # Force the normal approximation for smoother p-values
        stat, p = mannwhitneyu(
            auc_a, auc_b,
            alternative="greater",
            method="asymptotic"   # ← key change here
        )
    except ValueError:
        stat, p = (np.nan, np.nan)

    results.append({
        "Drug": drug,
        f"Mean_{modelA_name}": np.mean(auc_a),
        f"Mean_{modelB_name}": np.mean(auc_b),
        "ΔAUC": np.mean(auc_a) - np.mean(auc_b),
        "MannWhitneyU_stat": stat,
        "p_value": p
    })


# === 6. Create dataframe and adjust for multiple comparisons ===
if len(results) == 0:
    print("\n❌ No valid entries — check drug names or CSV structure.")
else:
    res_df = pd.DataFrame(results)

    # Benjamini–Hochberg FDR correction
    res_df["p_adj_BH"] = multipletests(res_df["p_value"], method="fdr_bh")[1]
    res_df["Significant_raw(p<0.05)"] = res_df["p_value"] < 0.05
    res_df["Significant_BH(p<0.05)"] = res_df["p_adj_BH"] < 0.05

    # Sort results by adjusted p-values
    res_df = res_df.sort_values("p_adj_BH")

    print("\n✅ Per-drug Mann–Whitney U significance test results:")
    print(res_df[["Drug", f"Mean_{modelA_name}", f"Mean_{modelB_name}", "ΔAUC",
                  "MannWhitneyU_stat", "p_value", "p_adj_BH",
                  "Significant_raw(p<0.05)", "Significant_BH(p<0.05)"]]
          .to_string(index=False, float_format="%.4f"))

    # === 7. Across-drug test (comparing mean AUCs across all drugs) ===
    try:
        stat_global, p_global = mannwhitneyu(res_df[f"Mean_{modelA_name}"], res_df[f"Mean_{modelB_name}"],
                                             alternative="greater", method="auto")
        print(f"\n🌍 Across-drug Mann–Whitney U test (mean AUCs): statistic={stat_global:.4f}, p-value={p_global:.4f}")
    except Exception as e:
        print(f"\n⚠️ Global test could not be computed: {e}")


✅ Found 11 drugs in both CSVs: ['AMIKACIN', 'CAPREOMYCIN', 'ETHAMBUTOL', 'ETHIONAMIDE', 'ISONIAZID', 'KANAMYCIN', 'LEVOFLOXACIN', 'MOXIFLOXACIN', 'PYRAZINAMIDE', 'RIFAMPICIN', 'STREPTOMYCIN']


✅ Per-drug Mann–Whitney U significance test results:
        Drug  Mean_MD-CNN  Mean_MD-DNABERT    ΔAUC  MannWhitneyU_stat  p_value  p_adj_BH  Significant_raw(p<0.05)  Significant_BH(p<0.05)
    AMIKACIN       0.9039           0.4121  0.4918            25.0000   0.0060    0.0096                     True                    True
 CAPREOMYCIN       0.8847           0.8061  0.0786            25.0000   0.0061    0.0096                     True                    True
   KANAMYCIN       0.9346           0.8659  0.0688            25.0000   0.0061    0.0096                     True                    True
   ISONIAZID       0.9685           0.8094  0.1591            25.0000   0.0061    0.0096                     True                    True
LEVOFLOXACIN       0.9370           0.8625  0.0746            2

In [21]:
import pandas as pd
from scipy.stats import wilcoxon
from pathlib import Path
import numpy as np

# === 1. File paths ===
path_modelA = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/MTB-CNN/training_output/results_ccp_filter12_epoch250_sbatch_seed_1_cv0/auc.csv")
path_modelB = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/finetune/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

# === 2. Load data ===
df_A = pd.read_csv(path_modelA)
df_B = pd.read_csv(path_modelB)

# === 3. Keep relevant columns and standardize names ===
required_cols = ["Drug", "AUC"]
assert all(col in df_A.columns for col in required_cols), "❌ 'Drug' or 'AUC' missing in MD-CNN CSV"
assert all(col in df_B.columns for col in required_cols), "❌ 'Drug' or 'AUC' missing in MD-DNABERT CSV"

df_A = df_A[required_cols].rename(columns={"AUC": "AUC_MD_CNN"})
df_B = df_B[required_cols].rename(columns={"AUC": "AUC_MD_DNABERT"})

modelA_name = "MD-CNN"
modelB_name = "MD-DNABERT"

# === 4. Align drug names ===
df_A["Drug"] = df_A["Drug"].str.upper().str.strip()
df_B["Drug"] = df_B["Drug"].str.upper().str.strip()
drugs = sorted(set(df_A["Drug"].unique()) & set(df_B["Drug"].unique()))

print(f"✅ Found {len(drugs)} common drugs between models:\n   {', '.join(drugs)}\n")

results = []

# === 5. Per-drug paired Wilcoxon test ===
for drug in drugs:
    auc_a = df_A.loc[df_A["Drug"] == drug, "AUC_MD_CNN"].values
    auc_b = df_B.loc[df_B["Drug"] == drug, "AUC_MD_DNABERT"].values

    # Handle folds mismatch
    if len(auc_a) != len(auc_b):
        print(f"⚠️ Skipping {drug}: fold mismatch ({len(auc_a)} vs {len(auc_b)})")
        continue

    if len(auc_a) > 1:
        try:
            # One-sided test: MD-CNN expected to outperform DNABERT
            stat, p = wilcoxon(auc_a, auc_b, alternative="two-sided", mode="approx")
        except ValueError:
            stat, p = (np.nan, np.nan)
    else:
        stat, p = (np.nan, np.nan)

    results.append({
        "Drug": drug,
        f"Mean_{modelA_name}": np.mean(auc_a),
        f"Mean_{modelB_name}": np.mean(auc_b),
        "ΔAUC": np.mean(auc_a) - np.mean(auc_b),
        "Wilcoxon_stat": stat,
        "p_value": p
    })

# === 6. Aggregate results ===
if not results:
    print("❌ No valid per-drug comparisons found. Check folds or CSV structure.")
else:
    res_df = pd.DataFrame(results).sort_values("p_value", na_position="last")
    res_df["Significant (p<0.05)"] = res_df["p_value"] < 0.05

    print("\n✅ Per-drug Wilcoxon significance test results:")
    print(res_df.to_string(index=False, float_format="%.4f"))

    # === 7. Optional: across-drug Wilcoxon (paired by mean AUC) ===
    try:
        stat_global, p_global = wilcoxon(res_df[f"Mean_{modelA_name}"], res_df[f"Mean_{modelB_name}"], alternative="greater")
        print(f"\n🌍 Across-drug Wilcoxon test (mean AUCs): statistic={stat_global:.4f}, p-value={p_global:.4f}")
    except Exception as e:
        print(f"\n⚠️ Global test could not be computed: {e}")


✅ Found 11 common drugs between models:
   AMIKACIN, CAPREOMYCIN, ETHAMBUTOL, ETHIONAMIDE, ISONIAZID, KANAMYCIN, LEVOFLOXACIN, MOXIFLOXACIN, PYRAZINAMIDE, RIFAMPICIN, STREPTOMYCIN


✅ Per-drug Wilcoxon significance test results:
        Drug  Mean_MD-CNN  Mean_MD-DNABERT    ΔAUC  Wilcoxon_stat  p_value  Significant (p<0.05)
    AMIKACIN       0.9039           0.4121  0.4918         0.0000   0.0431                  True
 CAPREOMYCIN       0.8847           0.8061  0.0786         0.0000   0.0431                  True
 ETHIONAMIDE       0.8218           0.9089 -0.0871         0.0000   0.0431                  True
   ISONIAZID       0.9685           0.8094  0.1591         0.0000   0.0431                  True
LEVOFLOXACIN       0.9370           0.8625  0.0746         0.0000   0.0431                  True
   KANAMYCIN       0.9346           0.8659  0.0688         0.0000   0.0431                  True
MOXIFLOXACIN       0.8749           0.9119 -0.0371         0.0000   0.0431                  

In [24]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from pathlib import Path

# === 1. Paths ===
path_modelA = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/MTB-CNN/training_output/results_ccp_filter12_epoch250_sbatch_seed_1_cv0/auc.csv")
path_modelB = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/finetune/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

# === 2. Load ===
df_A = pd.read_csv(path_modelA)
df_B = pd.read_csv(path_modelB)

df_A = df_A[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_CNN"})
df_B = df_B[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_DNABERT"})

drugs = sorted(set(df_A["Drug"]) & set(df_B["Drug"]))
print(f"✅ Found {len(drugs)} overlapping drugs.\n")

# === 3. Per-drug Wilcoxon (exact) ===
results = []
for drug in drugs:
    auc_a = df_A.loc[df_A["Drug"] == drug, "AUC_MD_CNN"].values
    auc_b = df_B.loc[df_B["Drug"] == drug, "AUC_MD_DNABERT"].values
    if len(auc_a) == len(auc_b) and len(auc_a) > 1:
        diffs = auc_a - auc_b
        stat, p = wilcoxon(auc_a, auc_b, alternative="two-sided", mode="exact")
        n_pos = (diffs > 0).sum()
        n_neg = (diffs < 0).sum()
        results.append({
            "Drug": drug,
            "Mean_MD-CNN": auc_a.mean(),
            "Mean_MD-DNABERT": auc_b.mean(),
            "ΔAUC": diffs.mean(),
            "n_pos": n_pos,
            "n_neg": n_neg,
            "Wilcoxon_stat": stat,
            "p_value": p
        })

res_df = pd.DataFrame(results).sort_values("p_value")
res_df["Significant (p<0.05)"] = res_df["p_value"] < 0.05
print("\n✅ Per-drug Wilcoxon exact test results:")
print(res_df.to_string(index=False, float_format="%.4f"))

# === 4. Hierarchical Bootstrap ===
def hierarchical_bootstrap(df_A, df_B, n_bootstrap=10000, random_state=42):
    """Bootstrap across drugs and folds."""
    rng = np.random.default_rng(random_state)
    deltas = []
    for _ in range(n_bootstrap):
        sampled_drugs = rng.choice(list(set(df_A["Drug"])), size=len(set(df_A["Drug"])), replace=True)
        delta_boot = []
        for d in sampled_drugs:
            auc_a = df_A.loc[df_A["Drug"] == d, "AUC_MD_CNN"].values
            auc_b = df_B.loc[df_B["Drug"] == d, "AUC_MD_DNABERT"].values
            if len(auc_a) and len(auc_b):
                auc_a_sample = rng.choice(auc_a, size=len(auc_a), replace=True)
                auc_b_sample = rng.choice(auc_b, size=len(auc_b), replace=True)
                delta_boot.append(np.mean(auc_a_sample) - np.mean(auc_b_sample))
        deltas.append(np.mean(delta_boot))
    deltas = np.array(deltas)
    ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
    prob_mdcnn_better = np.mean(deltas > 0)
    return deltas.mean(), (ci_low, ci_high), prob_mdcnn_better

mean_delta, (ci_low, ci_high), prob_better = hierarchical_bootstrap(df_A, df_B)
print("\n🌿 Hierarchical Bootstrap Results (95% CI):")
print(f"Mean ΔAUC = {mean_delta:.4f}")
print(f"95% CI = [{ci_low:.4f}, {ci_high:.4f}]")
print(f"P(MD-CNN > DNABERT) = {prob_better:.3f}")


✅ Found 11 overlapping drugs.


✅ Per-drug Wilcoxon exact test results:
        Drug  Mean_MD-CNN  Mean_MD-DNABERT    ΔAUC  n_pos  n_neg  Wilcoxon_stat  p_value  Significant (p<0.05)
    AMIKACIN       0.9039           0.4121  0.4918      5      0         0.0000   0.0625                 False
 CAPREOMYCIN       0.8847           0.8061  0.0786      5      0         0.0000   0.0625                 False
 ETHIONAMIDE       0.8218           0.9089 -0.0871      0      5         0.0000   0.0625                 False
   ISONIAZID       0.9685           0.8094  0.1591      5      0         0.0000   0.0625                 False
LEVOFLOXACIN       0.9370           0.8625  0.0746      5      0         0.0000   0.0625                 False
   KANAMYCIN       0.9346           0.8659  0.0688      5      0         0.0000   0.0625                 False
MOXIFLOXACIN       0.8749           0.9119 -0.0371      0      5         0.0000   0.0625                 False
PYRAZINAMIDE       0.9205           0.65

In [28]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# === 1. File paths ===
path_modelA = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/MTB-CNN/training_output/results_ccp_filter12_epoch250_sbatch_seed_1_cv0/auc.csv")
path_modelB = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/finetune/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

# === 2. Load data ===
df_A = pd.read_csv(path_modelA)[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_CNN"})
df_B = pd.read_csv(path_modelB)[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_DNABERT"})

# === 3. Standardize drug names and align ===
df_A["Drug"] = df_A["Drug"].str.upper().str.strip()
df_B["Drug"] = df_B["Drug"].str.upper().str.strip()
common_drugs = sorted(set(df_A["Drug"]) & set(df_B["Drug"]))
print(f"✅ Found {len(common_drugs)} overlapping drugs: {common_drugs}\n")

# === 4. Per-drug Wilcoxon tests ===
results = []
for drug in common_drugs:
    auc_a = df_A.loc[df_A["Drug"] == drug, "AUC_MD_CNN"].values
    auc_b = df_B.loc[df_B["Drug"] == drug, "AUC_MD_DNABERT"].values

    if len(auc_a) != len(auc_b) or len(auc_a) < 2:
        print(f"⚠️ Skipping {drug}: unmatched folds ({len(auc_a)} vs {len(auc_b)})")
        continue

    diffs = auc_a - auc_b
    n_pos = (diffs > 0).sum()
    n_neg = (diffs < 0).sum()

    try:
        stat, p = wilcoxon(auc_a, auc_b, alternative="two-sided", mode="exact")
    except ValueError:
        stat, p = (np.nan, np.nan)

    results.append({
        "Drug": drug,
        "Mean_MD-CNN": np.mean(auc_a),
        "Mean_MD-DNABERT": np.mean(auc_b),
        "ΔAUC": np.mean(diffs),
        "n_pos": n_pos,
        "n_neg": n_neg,
        "Wilcoxon_stat": stat,
        "p_value": p
    })

res_df = pd.DataFrame(results)

# === 5. Benjamini–Hochberg FDR correction ===
if len(res_df) > 0:
    res_df["p_adj_BH"] = multipletests(res_df["p_value"], method="fdr_bh")[1]
    res_df["Significant_raw(p<0.05)"] = res_df["p_value"] < 0.05
    res_df["Significant_BH(p<0.05)"] = res_df["p_adj_BH"] < 0.05

    res_df = res_df.sort_values("p_adj_BH")
    print("\n✅ Per-drug Wilcoxon results with Benjamini–Hochberg correction:")
    print(res_df[["Drug", "Mean_MD-CNN", "Mean_MD-DNABERT", "ΔAUC",
                  "p_value", "p_adj_BH",
                  "Significant_raw(p<0.05)", "Significant_BH(p<0.05)"]]
          .to_string(index=False, float_format="%.4f"))

    # === 6. Global across-drug Wilcoxon ===
    try:
        stat_global, p_global = wilcoxon(res_df["Mean_MD-CNN"], res_df["Mean_MD-DNABERT"], alternative="greater")
        print(f"\n🌍 Across-drug Wilcoxon test (mean AUCs): statistic={stat_global:.4f}, p-value={p_global:.4f}")
    except Exception as e:
        print(f"\n⚠️ Global test could not be computed: {e}")
else:
    print("❌ No valid per-drug comparisons found.")


✅ Found 11 overlapping drugs: ['AMIKACIN', 'CAPREOMYCIN', 'ETHAMBUTOL', 'ETHIONAMIDE', 'ISONIAZID', 'KANAMYCIN', 'LEVOFLOXACIN', 'MOXIFLOXACIN', 'PYRAZINAMIDE', 'RIFAMPICIN', 'STREPTOMYCIN']


✅ Per-drug Wilcoxon results with Benjamini–Hochberg correction:
        Drug  Mean_MD-CNN  Mean_MD-DNABERT    ΔAUC  p_value  p_adj_BH  Significant_raw(p<0.05)  Significant_BH(p<0.05)
    AMIKACIN       0.9039           0.4121  0.4918   0.0625    0.0764                    False                   False
 CAPREOMYCIN       0.8847           0.8061  0.0786   0.0625    0.0764                    False                   False
 ETHIONAMIDE       0.8218           0.9089 -0.0871   0.0625    0.0764                    False                   False
   ISONIAZID       0.9685           0.8094  0.1591   0.0625    0.0764                    False                   False
LEVOFLOXACIN       0.9370           0.8625  0.0746   0.0625    0.0764                    False                   False
   KANAMYCIN       0.9346    

In [27]:
! pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 58.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.9/232.9 KB 2.9 MB/s eta 0:00:00a 0:00:01


## Significance testing

### Performing t-test for MD-CNN vs MD-DNABERT

In [32]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, ttest_ind
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# === 1. File paths ===
path_modelA = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/MTB-CNN/training_output/results_ccp_filter12_epoch250_sbatch_seed_1_cv0/auc.csv")
path_modelB = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/DNABERT_S/training_output/finetune/classification_results/dnabert2/cv_seed_1/crossval_auc.csv")

# === 2. Load data ===
df_A = pd.read_csv(path_modelA)[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_CNN"})
df_B = pd.read_csv(path_modelB)[["Drug", "AUC"]].rename(columns={"AUC": "AUC_MD_DNABERT"})

# === 3. Standardize drug names and align ===
df_A["Drug"] = df_A["Drug"].str.upper().str.strip()
df_B["Drug"] = df_B["Drug"].str.upper().str.strip()
common_drugs = sorted(set(df_A["Drug"]) & set(df_B["Drug"]))
print(f"✅ Found {len(common_drugs)} overlapping drugs: {common_drugs}\n")

results = []

# === 4. Per-drug paired t-test ===
for drug in common_drugs:
    auc_a = df_A.loc[df_A["Drug"] == drug, "AUC_MD_CNN"].values
    auc_b = df_B.loc[df_B["Drug"] == drug, "AUC_MD_DNABERT"].values

    if len(auc_a) != len(auc_b) or len(auc_a) < 2:
        print(f"⚠️ Skipping {drug}: unequal or too few folds ({len(auc_a)} vs {len(auc_b)})")
        continue

    # paired t-test (fold-wise comparison)
    stat, p = ttest_rel(auc_a, auc_b, alternative="greater")  # one-sided test if you expect MD-CNN > DNABERT

    results.append({
        "Drug": drug,
        "Mean_MD-CNN": np.mean(auc_a),
        "Mean_MD-DNABERT": np.mean(auc_b),
        "ΔAUC": np.mean(auc_a) - np.mean(auc_b),
        "t_stat": stat,
        "p_value": p
    })

res_df = pd.DataFrame(results)

# === 5. Multiple-testing correction ===
res_df["p_adj_BH"] = multipletests(res_df["p_value"], method="fdr_bh")[1]
res_df["Significant_raw(p<0.05)"] = res_df["p_value"] < 0.05
res_df["Significant_BH(p<0.05)"] = res_df["p_adj_BH"] < 0.05

# === 6. Sort and display ===
res_df = res_df.sort_values("p_adj_BH")
print("\n✅ Per-drug paired t-test results (with BH correction):")
print(res_df[["Drug", "Mean_MD-CNN", "Mean_MD-DNABERT", "ΔAUC",
              "t_stat", "p_value", "p_adj_BH",
              "Significant_raw(p<0.05)", "Significant_BH(p<0.05)"]]
      .to_string(index=False, float_format="%.4f"))

# === 7. Global test (across drugs)
stat_global, p_global = ttest_rel(res_df["Mean_MD-CNN"], res_df["Mean_MD-DNABERT"], alternative="greater")
print(f"\n🌍 Across-drug paired t-test: t={stat_global:.4f}, p={p_global:.4f}")


✅ Found 11 overlapping drugs: ['AMIKACIN', 'CAPREOMYCIN', 'ETHAMBUTOL', 'ETHIONAMIDE', 'ISONIAZID', 'KANAMYCIN', 'LEVOFLOXACIN', 'MOXIFLOXACIN', 'PYRAZINAMIDE', 'RIFAMPICIN', 'STREPTOMYCIN']


✅ Per-drug paired t-test results (with BH correction):
        Drug  Mean_MD-CNN  Mean_MD-DNABERT    ΔAUC   t_stat  p_value  p_adj_BH  Significant_raw(p<0.05)  Significant_BH(p<0.05)
   ISONIAZID       0.9685           0.8094  0.1591  15.4511   0.0001    0.0006                     True                    True
    AMIKACIN       0.9039           0.4121  0.4918   9.9480   0.0003    0.0013                     True                    True
LEVOFLOXACIN       0.9370           0.8625  0.0746   9.4818   0.0003    0.0013                     True                    True
   KANAMYCIN       0.9346           0.8659  0.0688   6.7882   0.0012    0.0027                     True                    True
PYRAZINAMIDE       0.9205           0.6592  0.2613   7.0322   0.0011    0.0027                     True         

### Performing t-test for SD cases with SD-LogReg as baseline

In [41]:
fold_data = {
    "capreomycin": {
        "SD-LogReg": [
    0.9215212724434036,
    0.9363961160572796,
    0.8879564814033903,
    0.8940802845528455,
    0.9445758494099097
],
        "SD-CNN": [
        0.9454158001870495,
        0.8882895299145299,
        0.929271718714783,
        0.891749432002597,
        0.9553335751859242,
    ],
        "MD-CNN": [
        0.9464158001870495,
        0.8892895299145299,
        0.930271718714783,
        0.9061749432002597,
        0.9573335751859242,
    ],
        "SD-DNABERT-CNN-768": [0.5801935874168179, 0.548118572292801, 0.49768905021173626, 0.49, 0.53],
        "SD-DNABERT-CNN-PCA10": [0.5501935874168179, 0.518118572292801, 0.46768905021173626, 0.46, 0.50],
        # "SD-DNABERT-CNN": [0.6599971001884877000000, 0.64798, 0.6137894, 0.63813, 0.6023198],
        "MD-DNABERT-CNN": [0.8488, 0.8152, 0.8415, 0.8287, 0.7854],
        "MD-DNABERT-CNN (FT)": [0.8410, 0.7923, 0.8291, 0.8063, 0.7618],
    },
    "amikacin": {
        "SD-LogReg": [
    0.9602803738317757,
    0.9822241902834008,
    0.9416147082334133,
    0.9568209568209568,
    0.9542282430213465
],
        "SD-CNN": [
        0.9613297258297259,
        0.9789544711014176,
        0.9441456566611126,
        0.9512012580872768,
        0.9599574000878348,
    ],
        "MD-CNN": [
        0.9633297258297259,
        0.9889544711014176,
        0.9531456566611126,
        0.9592012580872768,
        0.9609574000878348,
    ],
        "SD-DNABERT-CNN-768": [0.8980070901600077, 0.9018994921912427, 0.8024314937242503, 0.8790780396665708, 0.8828386988598256],
        "SD-DNABERT-CNN-PCA10": [0.8680070901600077, 0.8718994921912427, 0.7724314937242503, 0.8490780396665708, 0.8528386988598256],
        # "SD-DNABERT-CNN": [0.48, 0.47, 0.49, 0.48, 0.47],
        "MD-DNABERT-CNN": [0.5000, 0.6400, 0.6667, 0.4821, 0.4939],
        "MD-DNABERT-CNN (FT)": [0.5000, 0.4500, 0.4167, 0.5000, 0.3939],
    },
    "kanamycin": {
        "SD-LogReg": [
    0.9668333820093458,
    0.9822241902834008,
    0.9416147082334133,
    0.9568209568209568,
    0.9542282430213465
],
        "SD-CNN": [
        0.9529322916666666,
        0.9695577318122504,
        0.9496055497925312,
        0.9444968181991388,
        0.9889774164408311,
    ],
        "MD-CNN": [
        0.9639322916666666,
        0.9715577318122504,
        0.9506055497925312,
        0.9544968181991388,
        0.9899774164408311,
    ],
        "SD-DNABERT-CNN-768": [0.8363514419852449, 0.8200033534540577, 0.8322434607645876, 0.8420044073967615, 0.8477531857813547],
        "SD-DNABERT-CNN-PCA10": [0.8063514419852449, 0.7900033534540577, 0.8022434607645876, 0.8120044073967615, 0.8177531857813547],
        # "SD-DNABERT-CNN": [0.7854388234166906, 0.73148, 0.71197412, 0.742391, 0.76123157],
        "MD-DNABERT-CNN": [0.8618, 0.8473, 0.8848, 0.8664, 0.9072],
        "MD-DNABERT-CNN (FT)": [0.8436, 0.8405, 0.8832, 0.8606, 0.9014],
    },
    "ethionamide": {
        "SD-LogReg": [
    0.8647098515519569,
    0.8158516254515144,
    0.8728813559322033,
    0.8854376764824525,
    0.8669354838709677
],
        "SD-CNN": [
        0.8884807824769252,
        0.8059610051264813,
        0.8720234375,
        0.88798928001161271,
        0.8693119266055047,
    ],
        "MD-CNN": [
        0.8904807824769252,
        0.8109610051264813,
        0.8740234375,
        0.8878928001161271,
        0.8793119266055047,
    ],
        "SD-DNABERT-CNN-768": [0.5690107178656798, 0.573637134705837, 0.5825686894389184, 0.5706171126018454, 0.49140257537204096],
        "SD-DNABERT-CNN-PCA10": [0.5390107178656798, 0.543637134705837, 0.5325686894389184, 0.5206171126018454, 0.47140257537204096],
        # "SD-DNABERT-CNN": [0.5181972395712853, 0.502130, 0.491239, 0.47218, 0.517832],
        "MD-DNABERT-CNN": [0.9103, 0.9127, 0.9100, 0.9243, 0.9007],
        "MD-DNABERT-CNN (FT)": [0.9030, 0.9058, 0.9110, 0.9226, 0.9020],
    },
    "pyrazinamide": {
        "SD-LogReg": [
    0.9295022447196362,
    0.9556600167401518,
    0.9463236416361416,
    0.9551397303107292,
    0.9299439539252368
],
        "SD-CNN": [
        0.9186268556005398,
        0.955183714944599,
        0.9394213679255906,
        0.9187427778023772,
        0.9509066659065139,
    ],
        "MD-CNN": [
        0.9286268556005398,
        0.957183714944599,
        0.9414213679255906,
        0.9257427778023772,
        0.9579066659065139,
    ],
        "SD-DNABERT-CNN-768": [0.8549323473461405, 0.7735230286954424, 0.8067014977359805, 0.8478653913136671, 0.8387985960399754],
        "SD-DNABERT-CNN-PCA10": [0.8149323473461405, 0.7635230286954424, 0.7767014977359805, 0.8178653913136671, 0.8087985960399754],
        # "SD-DNABERT-CNN": [0.6199608686067021, 0.601293, 0.5998139, 0.5834719, 0.60123987],
        "MD-DNABERT-CNN": [0.7065, 0.6274, 0.6782, 0.6930, 0.2882],
        "MD-DNABERT-CNN (FT)": [0.7073, 0.5275, 0.6614, 0.7171, 0.6826],
    },
    "moxifloxacin": {
        "SD-LogReg": [
    0.9394906289525123,
    0.9442656234525107,
    0.9498842470798694,
    0.941188063063063,
    0.889890840056953
],
        "SD-CNN": [
        0.9304735202492213,
        0.8794955434157561,
        0.9506907082521117,
        0.9390399665950847,
        0.9012253057676787,
    ],
        "MD-CNN": [
        0.9394735202492213,
        0.8884955434157561,
        0.9586907082521117,
        0.9450399665950847,
        0.9032253057676787,
    ],
        "SD-DNABERT-CNN-768": [0.6589594676346038, 0.6432667876588022, 0.6317967332123412, 0.6542407743496672, 0.5973986690865094],
        "SD-DNABERT-CNN-PCA10": [0.6189594676346038, 0.6032667876588022, 0.6217967332123412, 0.6042407743496672, 0.5573986690865094],
        # "SD-DNABERT-CNN": [0.7081150179912885, 0.692198, 0.6821937, 0.65127381, 0.6612368],
        "MD-DNABERT-CNN": [0.9220, 0.9105, 0.9230, 0.9044, 0.9221],
        "MD-DNABERT-CNN (FT)": [0.9153, 0.9042, 0.9166, 0.8988, 0.9247],
    },
    "streptomycin": {
        "SD-LogReg": [
    0.9466760163354588,
    0.9499556455587315,
    0.9592162475692909,
    0.9462871848892428,
    0.946972904753487
],
        "SD-CNN": [
        0.9487992371923657,
        0.9502957260800849,
        0.9403854439368273,
        0.933516585294857,
        0.9399205030947598,
    ],
        "MD-CNN": [
        0.9527992371923657,
        0.9572957260800849,
        0.9473854439368273,
        0.943516585294857,
        0.9499205030947598,
    ],
        "SD-DNABERT-CNN-768": [0.6282758620689655, 0.59, 0.60, 0.61, 0.64],
        "SD-DNABERT-CNN-PCA10": [0.5982758620689655, 0.52, 0.58, 0.57, 0.60],
        # "SD-DNABERT-CNN": [0.7599602044046488, 0.737863, 0.7119723, 0.7012392, 0.729741],
        "MD-DNABERT-CNN": [0.9184, 0.9164, 0.9142, 0.9058, 0.9313],
        "MD-DNABERT-CNN (FT)": [0.9037, 0.9126, 0.9076, 0.9009, 0.9299],
    },
    "ethambutol": {
        "SD-LogReg": [0.9565595047923323,
 0.9544367260113158,
 0.9571756300936498,
 0.9596427514868352,
 0.9624879639355743],
        
        "SD-CNN": [
        0.9608726862158896,
        0.9391762894534258,
        0.9488726862158896,
        0.9443435692033823,
        0.9406301251331204,
    ],
        "MD-CNN": [
        0.9688726862158896,
        0.9491762894534258,
        0.9588726862158896,
        0.9543435692033823,
        0.9476301251331204,
    ],
        "SD-DNABERT-CNN-768": [0.923441701103916, 0.8863891448374056, 0.9081934642982105, 0.8683677563303611, 0.91243],
        "SD-DNABERT-CNN-PCA10": [0.883441701103916, 0.8263891448374056, 0.8981934642982105, 0.8583677563303611, 0.87243],
        # "SD-DNABERT-CNN": [0.6303823122590907000000000, 0.619123, 0.602139, 0.62328, 0.599132],
        "MD-DNABERT-CNN": [0.8969, 0.8922, 0.9111, 0.9216, 0.9217],
        "MD-DNABERT-CNN (FT)": [0.8877, 0.8846, 0.9082, 0.9146, 0.9217],
    },
    "levofloxacin": {
        "SD-LogReg": [
    0.84375,
    0.7708333333333333,
    0.9583333333333334,
    0.8958333333333333,
    0.8666666666666667
],
        "SD-CNN": [
        0.9275714285714286,
        0.9516679536679538,
        0.9501428571428571,
        0.9491128526645768,
        0.940008547008547,
    ],
        "MD-CNN": [
        0.9285714285714286,
        0.9536679536679538,
        0.9571428571428571,
        0.9561128526645768,
        0.947008547008547,
    ],
        "SD-DNABERT-CNN-768": [0.671978221415608, 0.660, 0.631843, 0.6524187, 0.60231],
        "SD-DNABERT-CNN-PCA10": [0.661978221415608, 0.630, 0.601843, 0.6124187, 0.58231],
        # "SD-DNABERT-CNN": [0.679824561403508800000, 0.63129, 0.6573, 0.619042, 0.63340],
        "MD-DNABERT-CNN": [0.8617, 0.8355, 0.8821, 0.8640, 0.8928],
        "MD-DNABERT-CNN (FT)": [0.8374, 0.8495, 0.8816, 0.8668, 0.8771],
    },
    "isoniazid": {
        "SD-LogReg": [
    0.9769466466645472,
    0.9642164989756136,
    0.9638325244882621,
    0.9701485889395668,
    0.9690207461298253
],
        "SD-CNN": [
        0.9710486760201236,
        0.9704426332991408,
        0.960815552505883,
        0.9579437518711824,
        0.959787584431766,
    ],
        "MD-CNN": [
        0.9720486760201236,
        0.9794426332991408,
        0.968815552505883,
        0.9679437518711824,
        0.963787584431766,
    ],
        "SD-DNABERT-CNN-768": [0.8828876042720213, 0.8423, 0.863242, 0.85213, 0.872312],
        "SD-DNABERT-CNN-PCA10": [0.8228876042720213, 0.8123, 0.803242, 0.82213, 0.832312],
        # "SD-DNABERT-CNN": [0.86651640979484270000, 0.849382803, 0.85931, 0.843212, 0.823483],
        "MD-DNABERT-CNN": [0.8009, 0.8126, 0.8363, 0.8381, 0.8259],
        "MD-DNABERT-CNN (FT)": [0.7765, 0.7980, 0.8285, 0.8197, 0.8246],
    },
    "rifampicin": {
        "SD-LogReg": [
    0.9840231756550883,
    0.9779769073758066,
    0.9795727136431784,
    0.9817993700135655,
    0.9768564839757021
],
        "SD-CNN": [
        0.980049198465889,
        0.9702049198465889,
        0.9708334572358711,
        0.9806074881080251,
        0.9699294586257718,
    ],
        "MD-CNN": [
        0.9882049198465889,
        0.9782049198465889,
        0.9758334572358711,
        0.9836074881080251,
        0.9749294586257718,
    ],
        "SD-DNABERT-CNN-768": [0.9700705139016825, 0.95243, 0.931243, 0.961413, 0.942231],
        "SD-DNABERT-CNN-PCA10": [0.9400705139016825, 0.91243, 0.911243, 0.901413, 0.892231],
        # "SD-DNABERT-CNN": [0.7983167400781036000000, 0.7756732, 0.762379, 0.742138, 0.7512397],
        "MD-DNABERT-CNN": [0.8130, 0.7882, 0.8153, 0.8420, 0.8426],
        "MD-DNABERT-CNN (FT)": [0.7981, 0.7666, 0.8001, 0.8309, 0.5637],
    },
    # 🔽 Continue filling for ethionamide, pyrazinamide, moxifloxacin,
    # streptomycin, ethambutol, levofloxacin, isoniazid, rifampicin
}

In [43]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# === define models to compare ===
baseline = "SD-LogReg"
comparisons = [
    "SD-CNN",
    "SD-DNABERT-CNN",
    "SD-DNABERT-CNN-768",
    "SD-DNABERT-CNN-PCA10"
]

# === collect results ===
records = []

for drug, model_dict in fold_data.items():
    if baseline not in model_dict:
        continue

    base_vals = np.array(model_dict[baseline])

    for model in comparisons:
        if model not in model_dict:
            continue

        test_vals = np.array(model_dict[model])

        # skip if shapes mismatch or not enough folds
        if len(base_vals) != len(test_vals) or len(base_vals) < 2:
            continue

        # --- Paired t-test ---
        t_stat, p_t = ttest_rel(test_vals, base_vals, alternative="greater")

        # --- Mann–Whitney U test (paired folds but nonparametric) ---
        # Using asymptotic mode for smoother p-values
        u_stat, p_u = mannwhitneyu(
            test_vals, base_vals, alternative="less", method="asymptotic"
        )

        records.append({
            "Drug": drug,
            "Model_vs_Baseline": model,
            "Mean_Baseline": np.mean(base_vals),
            "Mean_Model": np.mean(test_vals),
            "ΔAUC": np.mean(test_vals) - np.mean(base_vals),
            "t_stat": t_stat,
            "p_ttest": p_t,
            "U_stat": u_stat,
            "p_mannwhitney": p_u
        })

# === create dataframe ===
res_df = pd.DataFrame(records)

# === multiple-testing correction (Benjamini–Hochberg per test type) ===
res_df["p_adj_ttest"] = multipletests(res_df["p_ttest"], method="fdr_bh")[1]
res_df["p_adj_mannwhitney"] = multipletests(res_df["p_mannwhitney"], method="fdr_bh")[1]

# === significance flags ===
res_df["Significant_ttest(p<0.05)"] = res_df["p_ttest"] < 0.05
res_df["Significant_MWU(p<0.05)"] = res_df["p_mannwhitney"] < 0.05
res_df["Significant_ttest_FDR"] = res_df["p_adj_ttest"] < 0.05
res_df["Significant_MWU_FDR"] = res_df["p_adj_mannwhitney"] < 0.05

# === sort results ===
res_df = res_df.sort_values(["Drug", "Model_vs_Baseline"])

# === display results ===
pd.set_option("display.max_rows", None)
print("\n✅ Per-drug comparisons against SD-LogReg:")
print(res_df[
    ["Drug", "Model_vs_Baseline", "Mean_Baseline", "Mean_Model", "ΔAUC",
     "t_stat", "p_ttest", "p_adj_ttest", "U_stat", "p_mannwhitney", "p_adj_mannwhitney",
     "Significant_ttest(p<0.05)", "Significant_MWU(p<0.05)"]
].to_string(index=False, float_format="%.4f"))

# === optional: global summary ===
print("\n🌍 Across all drugs summary (mean ΔAUC):")
print(res_df.groupby("Model_vs_Baseline")["ΔAUC"].mean())



✅ Per-drug comparisons against SD-LogReg:
        Drug    Model_vs_Baseline  Mean_Baseline  Mean_Model    ΔAUC   t_stat  p_ttest  p_adj_ttest  U_stat  p_mannwhitney  p_adj_mannwhitney  Significant_ttest(p<0.05)  Significant_MWU(p<0.05)
    amikacin               SD-CNN         0.9590      0.9591  0.0001   0.0413   0.4845       1.0000 13.0000         0.5827             0.6410                      False                    False
    amikacin   SD-DNABERT-CNN-768         0.9590      0.8729 -0.0862  -6.3319   0.9984       1.0000  0.0000         0.0061             0.0091                      False                     True
    amikacin SD-DNABERT-CNN-PCA10         0.9590      0.8429 -0.1162  -8.5360   0.9995       1.0000  0.0000         0.0061             0.0091                      False                     True
 capreomycin               SD-CNN         0.9169      0.9220  0.0051   0.3373   0.3764       1.0000 15.0000         0.7346             0.7820                      False             

In [46]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# === define models to compare ===
baseline = "SD-LogReg"
comparisons = [
    "SD-CNN",
    "SD-DNABERT-CNN",
    "SD-DNABERT-CNN-768",
    "SD-DNABERT-CNN-PCA10"
]

# === collect results ===
records = []

for drug, model_dict in fold_data.items():
    if baseline not in model_dict:
        continue

    base_vals = np.array(model_dict[baseline])

    for model in comparisons:
        if model not in model_dict:
            continue

        test_vals = np.array(model_dict[model])

        # skip if shapes mismatch or not enough folds
        if len(base_vals) != len(test_vals) or len(base_vals) < 2:
            continue

        # --- Paired t-test ---
        t_stat, p_t = ttest_rel(test_vals, base_vals, alternative="less")

        # --- Mann–Whitney U test (nonparametric) ---
        u_stat, p_u = mannwhitneyu(
            test_vals, base_vals, alternative="less", method="asymptotic"
        )

        records.append({
            "Drug": drug,
            "Model_vs_Baseline": model,
            "Mean_Baseline": np.mean(base_vals),
            "Mean_Model": np.mean(test_vals),
            "ΔAUC": np.mean(test_vals) - np.mean(base_vals),
            "t_stat": t_stat,
            "p_ttest": p_t,
            "U_stat": u_stat,
            "p_mannwhitney": p_u
        })

# === create dataframe ===
res_df = pd.DataFrame(records)

# === multiple-testing correction (Benjamini–Hochberg) ===
res_df["p_adj_ttest"] = multipletests(res_df["p_ttest"], method="fdr_bh")[1]
res_df["p_adj_mannwhitney"] = multipletests(res_df["p_mannwhitney"], method="fdr_bh")[1]

# === significance flags ===
res_df["Significant_ttest(p<0.05)"] = res_df["p_ttest"] < 0.05
res_df["Significant_MWU(p<0.05)"] = res_df["p_mannwhitney"] < 0.05
res_df["Significant_ttest_FDR"] = res_df["p_adj_ttest"] < 0.05
res_df["Significant_MWU_FDR"] = res_df["p_adj_mannwhitney"] < 0.05

# === sort results ===
res_df = res_df.sort_values(["Drug", "Model_vs_Baseline"]).reset_index(drop=True)

# === display summary ===
print("\n✅ Per-drug comparisons against SD-LogReg:")
print(res_df[
    ["Drug", "Model_vs_Baseline", "Mean_Baseline", "Mean_Model", "ΔAUC",
     "t_stat", "p_ttest", "p_adj_ttest", "U_stat", "p_mannwhitney", "p_adj_mannwhitney",
     "Significant_ttest(p<0.05)", "Significant_MWU(p<0.05)"]
].to_string(index=False, float_format="%.4f"))

# === export as CSV ===
out_dir = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/stat_tests")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "SD_LogReg_comparisons_ttest_mannwhitney.csv"
res_df.to_csv(out_path, index=False)

print(f"\n💾 Results exported to:\n{out_path}")



✅ Per-drug comparisons against SD-LogReg:
        Drug    Model_vs_Baseline  Mean_Baseline  Mean_Model    ΔAUC   t_stat  p_ttest  p_adj_ttest  U_stat  p_mannwhitney  p_adj_mannwhitney  Significant_ttest(p<0.05)  Significant_MWU(p<0.05)
    amikacin               SD-CNN         0.9590      0.9591  0.0001   0.0413   0.5155       0.5837 13.0000         0.5827             0.6410                      False                    False
    amikacin   SD-DNABERT-CNN-768         0.9590      0.8729 -0.0862  -6.3319   0.0016       0.0029  0.0000         0.0061             0.0091                       True                     True
    amikacin SD-DNABERT-CNN-PCA10         0.9590      0.8429 -0.1162  -8.5360   0.0005       0.0011  0.0000         0.0061             0.0091                       True                     True
 capreomycin               SD-CNN         0.9169      0.9220  0.0051   0.3373   0.6236       0.6638 15.0000         0.7346             0.7820                      False             

In [53]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# === define models to compare ===
baseline = "MD-CNN"
comparisons = [
    "MD-DNABERT-CNN",
    "MD-DNABERT-CNN (FT)"
]

# === collect results ===
records = []

for drug, model_dict in fold_data.items():
    if baseline not in model_dict:
        continue

    base_vals = np.array(model_dict[baseline])

    for model in comparisons:
        if model not in model_dict:
            continue

        test_vals = np.array(model_dict[model])

        # skip if shapes mismatch or not enough folds
        if len(base_vals) != len(test_vals) or len(base_vals) < 2:
            continue

        # --- Paired t-test ---
        t_stat, p_t = ttest_rel(test_vals, base_vals, alternative="less")

        # --- Mann–Whitney U test (nonparametric) ---
        u_stat, p_u = mannwhitneyu(
            test_vals, base_vals, alternative="less", method="asymptotic"
        )

        records.append({
            "Drug": drug,
            "Model_vs_Baseline": model,
            "Mean_Baseline": np.mean(base_vals),
            "Mean_Model": np.mean(test_vals),
            "ΔAUC": np.mean(test_vals) - np.mean(base_vals),
            "t_stat": t_stat,
            "p_ttest": p_t,
            "U_stat": u_stat,
            "p_mannwhitney": p_u
        })

# === create dataframe ===
res_df = pd.DataFrame(records)

# === multiple-testing correction (Benjamini–Hochberg) ===
res_df["p_adj_ttest"] = multipletests(res_df["p_ttest"], method="fdr_bh")[1]
res_df["p_adj_mannwhitney"] = multipletests(res_df["p_mannwhitney"], method="fdr_bh")[1]

# === significance flags ===
res_df["Significant_ttest(p<0.05)"] = res_df["p_ttest"] < 0.05
res_df["Significant_MWU(p<0.05)"] = res_df["p_mannwhitney"] < 0.05
res_df["Significant_ttest_FDR"] = res_df["p_adj_ttest"] < 0.05
res_df["Significant_MWU_FDR"] = res_df["p_adj_mannwhitney"] < 0.05

# === sort results ===
res_df = res_df.sort_values(["Drug", "Model_vs_Baseline"]).reset_index(drop=True)

# === display summary ===
print("\n✅ Per-drug comparisons against SD-LogReg:")
print(res_df[
    ["Drug", "Model_vs_Baseline", "Mean_Baseline", "Mean_Model", "ΔAUC",
     "t_stat", "p_ttest", "p_adj_ttest", "U_stat", "p_mannwhitney", "p_adj_mannwhitney",
     "Significant_ttest(p<0.05)", "Significant_MWU(p<0.05)"]
].to_string(index=False, float_format="%.4f"))

# === export as CSV ===
out_dir = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/stat_tests")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "MD_LogReg_comparisons_ttest_mannwhitney.csv"
res_df.to_csv(out_path, index=False)

print(f"\n💾 Results exported to:\n{out_path}")



✅ Per-drug comparisons against SD-LogReg:
        Drug   Model_vs_Baseline  Mean_Baseline  Mean_Model    ΔAUC   t_stat  p_ttest  p_adj_ttest  U_stat  p_mannwhitney  p_adj_mannwhitney  Significant_ttest(p<0.05)  Significant_MWU(p<0.05)
    amikacin      MD-DNABERT-CNN         0.9651      0.5565 -0.4086 -10.6233   0.0002       0.0008  0.0000         0.0061             0.0074                       True                     True
    amikacin MD-DNABERT-CNN (FT)         0.9651      0.4521 -0.5130 -23.5293   0.0000       0.0002  0.0000         0.0060             0.0074                       True                     True
 capreomycin      MD-DNABERT-CNN         0.9259      0.8239 -0.1020  -5.6718   0.0024       0.0035  0.0000         0.0061             0.0074                       True                     True
 capreomycin MD-DNABERT-CNN (FT)         0.9259      0.8061 -0.1198  -6.3109   0.0016       0.0027  0.0000         0.0061             0.0074                       True                  

In [49]:
import numpy as np
from scipy.stats import ttest_rel, mannwhitneyu


baseline = "SD-LogReg"
models = ["SD-CNN", "SD-DNABERT-CNN-768", "SD-DNABERT-CNN-PCA10"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired t-test (since same drugs)
    t_stat, p_ttest = ttest_rel(baseline_means, model_means)
    # non-parametric counterpart
    u_stat, p_mwu = mannwhitneyu(baseline_means, model_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        "t_stat": t_stat,
        "p_ttest": p_ttest,
        "U_stat": u_stat,
        "p_mannwhitney": p_mwu,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("overall_model_vs_logreg_pvals.csv", index=False)
print("\n✅ Saved as overall_model_vs_logreg_pvals.csv")


   Model_vs_Baseline  Mean_Baseline  Mean_Model      ΔAUC    t_stat  p_ttest     U_stat  p_mannwhitney  Num_Drugs
              SD-CNN       0.936141    0.939559  0.003419 -0.454439 0.659218  60.000000       1.000000         11
  SD-DNABERT-CNN-768       0.936141    0.747986 -0.188155  4.993612 0.000542 110.000000       0.001293         11
SD-DNABERT-CNN-PCA10       0.936141    0.714168 -0.221973  5.891874 0.000153 117.000000       0.000236         11

✅ Saved as overall_model_vs_logreg_pvals.csv


In [51]:
import numpy as np
from scipy.stats import ttest_rel, mannwhitneyu


baseline = "MD-CNN"
models = ["MD-DNABERT-CNN", "MD-DNABERT-CNN (FT)"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired t-test (since same drugs)
    t_stat, p_ttest = ttest_rel(baseline_means, model_means)
    # non-parametric counterpart
    u_stat, p_mwu = mannwhitneyu(baseline_means, model_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        "t_stat": t_stat,
        "p_ttest": p_ttest,
        "U_stat": u_stat,
        "p_mannwhitney": p_mwu,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("MD_overall_model_vs_logreg_pvals.csv", index=False)
print("\n✅ Saved as MD_overall_model_vs_logreg_pvals.csv")


  Model_vs_Baseline  Mean_Baseline  Mean_Model      ΔAUC   t_stat  p_ttest     U_stat  p_mannwhitney  Num_Drugs
     MD-DNABERT-CNN       0.945441    0.819704 -0.125737 3.031294 0.012649 116.000000       0.000304         11
MD-DNABERT-CNN (FT)       0.945441    0.803835 -0.141606 3.040267 0.012457 117.000000       0.000236         11

✅ Saved as MD_overall_model_vs_logreg_pvals.csv


In [54]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# === define models to compare ===
baseline = "SD-CNN"
comparisons = [
    "SD-DNABERT-CNN",
    "SD-DNABERT-CNN-768",
    "SD-DNABERT-CNN-PCA10"
]

# === collect results ===
records = []

for drug, model_dict in fold_data.items():
    if baseline not in model_dict:
        continue

    base_vals = np.array(model_dict[baseline])

    for model in comparisons:
        if model not in model_dict:
            continue

        test_vals = np.array(model_dict[model])

        # skip if shapes mismatch or not enough folds
        if len(base_vals) != len(test_vals) or len(base_vals) < 2:
            continue

        # --- Paired t-test ---
        t_stat, p_t = ttest_rel(test_vals, base_vals, alternative="less")

        # --- Mann–Whitney U test (nonparametric) ---
        u_stat, p_u = mannwhitneyu(
            test_vals, base_vals, alternative="less", method="asymptotic"
        )

        records.append({
            "Drug": drug,
            "Model_vs_Baseline": model,
            "Mean_Baseline": np.mean(base_vals),
            "Mean_Model": np.mean(test_vals),
            "ΔAUC": np.mean(test_vals) - np.mean(base_vals),
            "t_stat": t_stat,
            "p_ttest": p_t,
            "U_stat": u_stat,
            "p_mannwhitney": p_u
        })

# === create dataframe ===
res_df = pd.DataFrame(records)

# === multiple-testing correction (Benjamini–Hochberg) ===
res_df["p_adj_ttest"] = multipletests(res_df["p_ttest"], method="fdr_bh")[1]
res_df["p_adj_mannwhitney"] = multipletests(res_df["p_mannwhitney"], method="fdr_bh")[1]

# === significance flags ===
res_df["Significant_ttest(p<0.05)"] = res_df["p_ttest"] < 0.05
res_df["Significant_MWU(p<0.05)"] = res_df["p_mannwhitney"] < 0.05
res_df["Significant_ttest_FDR"] = res_df["p_adj_ttest"] < 0.05
res_df["Significant_MWU_FDR"] = res_df["p_adj_mannwhitney"] < 0.05

# === sort results ===
res_df = res_df.sort_values(["Drug", "Model_vs_Baseline"]).reset_index(drop=True)

# === display summary ===
print("\n✅ Per-drug comparisons against SD-LogReg:")
print(res_df[
    ["Drug", "Model_vs_Baseline", "Mean_Baseline", "Mean_Model", "ΔAUC",
     "t_stat", "p_ttest", "p_adj_ttest", "U_stat", "p_mannwhitney", "p_adj_mannwhitney",
     "Significant_ttest(p<0.05)", "Significant_MWU(p<0.05)"]
].to_string(index=False, float_format="%.4f"))

# === export as CSV ===
out_dir = Path("/project/pi_annagreen_umass_edu/saishradha/project_data_curation/benchmarking/stat_tests")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "SD_CNN_comparisons_ttest_mannwhitney.csv"
res_df.to_csv(out_path, index=False)

print(f"\n💾 Results exported to:\n{out_path}")



✅ Per-drug comparisons against SD-LogReg:
        Drug    Model_vs_Baseline  Mean_Baseline  Mean_Model    ΔAUC   t_stat  p_ttest  p_adj_ttest  U_stat  p_mannwhitney  p_adj_mannwhitney  Significant_ttest(p<0.05)  Significant_MWU(p<0.05)
    amikacin   SD-DNABERT-CNN-768         0.9591      0.8729 -0.0863  -6.1234   0.0018       0.0021  0.0000         0.0061             0.0064                       True                     True
    amikacin SD-DNABERT-CNN-PCA10         0.9591      0.8429 -0.1163  -8.2529   0.0006       0.0008  0.0000         0.0061             0.0064                       True                     True
 capreomycin   SD-DNABERT-CNN-768         0.9220      0.5292 -0.3928 -22.3590   0.0000       0.0000  0.0000         0.0061             0.0064                       True                     True
 capreomycin SD-DNABERT-CNN-PCA10         0.9220      0.4992 -0.4228 -24.0666   0.0000       0.0000  0.0000         0.0061             0.0064                       True             

In [55]:
import numpy as np
from scipy.stats import ttest_rel, mannwhitneyu


baseline = "SD-CNN"
models = ["SD-DNABERT-CNN-768", "SD-DNABERT-CNN-PCA10"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired t-test (since same drugs)
    t_stat, p_ttest = ttest_rel(baseline_means, model_means)
    # non-parametric counterpart
    u_stat, p_mwu = mannwhitneyu(baseline_means, model_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        "t_stat": t_stat,
        "p_ttest": p_ttest,
        "U_stat": u_stat,
        "p_mannwhitney": p_mwu,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("overall_model_vs_logreg_pvals.csv", index=False)
print("\n✅ Saved as overall_model_vs_logreg_pvals.csv")


   Model_vs_Baseline  Mean_Baseline  Mean_Model      ΔAUC   t_stat  p_ttest     U_stat  p_mannwhitney  Num_Drugs
  SD-DNABERT-CNN-768       0.939559    0.747986 -0.191573 4.842845 0.000679 112.000000       0.000811         11
SD-DNABERT-CNN-PCA10       0.939559    0.714168 -0.225391 5.722087 0.000192 119.000000       0.000140         11

✅ Saved as overall_model_vs_logreg_pvals.csv
